# Chapter 6 Exercise — Coffee Maker Recursive Decomposition

Decompose `BrewUnit`'s own `applyWater` step one level deeper, then record
the two judgment sites that decomposition raises and a stopping judgment.

## Problem

Your model from Chapter 5 nests `applyWater` inside `Brew` (Chapter 4) and
allocates it, at the usage level, to `brewUnit` (Chapter 5). This chapter
asks: what does `ApplyWater` itself do internally, and what carries it?
Mirror main Chapter 6's own `GenerateHeat`/`HeatGenerator`/`HeatingAssembly`
chain exactly, one level down, for the water-moving step instead of the
heat-generating one.

Work through these steps:

1. Add a level-2 function nested inside `ApplyWater`, `MoveWater`, mirroring
   `GenerateHeat`'s own signature-only pattern nested inside `ApplyHeat`
   (typed flows only, no mechanism, no energy or material form committed:
   `in waterIn : ISQ::MassValue[0..*]; out waterOut : ISQ::MassValue;`).
   `ApplyWater` cannot have a nested step added in a separate statement, so
   its full declaration is reprinted with `first start; then action
   moveWater : MoveWater { in waterIn = ApplyWater::water; } then done;`
   added, mirroring the idiom main Chapter 6 notebook 01 uses for
   `ApplyHeat`. Then add `WaterMover`, an abstract logical carrier one level
   below `BrewUnit`, mirroring `HeatGenerator` exactly: `perform action
   moveWater : MoveWater;`, a port (reuse Chapter 5's own `WaterPort`,
   conjugated: `port waterIn : ~WaterPort;`, mirroring `HeatGenerator`'s own
   `~EnergyPort`-style port), and an unbound `attribute throughput :
   ISQ::MassFlowRateValue;` — no default, the same uncommitted design-space
   slot `HeatGenerator::power` is.
2. Add `part def BrewAssembly :> BrewUnit`, mirroring `HeatingAssembly :>
   HeatingSystem`. Unlike `HeatingSystem`, `BrewUnit` (Chapter 1) was never
   given its own direct `perform action applyWater : ApplyWater;`, so
   `BrewAssembly` declares that perform itself, then composes `part mover :
   WaterMover;` and nests the usage-level allocation `allocation
   moverAllocation allocate applyWater.moveWater to mover;`, mirroring
   `heatGenAllocation` exactly. Also add `requirement def BrewReq` with
   `subject wm : WaterMover;` (not `BrewUnit` — mirroring
   `HeatGenerationReq`'s subject being `HeatGenerator`, not `HeatingSystem`)
   and `require constraint { wm.throughput >= 0.003 ['kg⋅s⁻¹'] }`, plus a
   `requirement brewReq : BrewReq;` usage. Give `BrewReq` a `doc` stating a
   component throughput rating threshold and naming no stakeholder
   acceptance criterion, mirroring `HeatGenerationReq`'s own doc exactly
   (a rating, "not yet derived from a stated measure of effectiveness...
   Recorded openly, not faked"): `AC-C06-EX` below cites this doc as
   evidence, so its wording matters, even though the framing question
   itself stays open for you to argue.
3. Write an `asserted_context` record `AC-C06-EX` framing what kind of
   measure `brewReq` constrains: a measure of effectiveness or a measure
   of performance — mirroring `AC-C06`'s own framing of `heatGenerationReq`
   in main Chapter 6 notebook 02. This is a genuine judgment call for you to
   make and justify: is the throughput threshold a component-level
   engineering rating, or a user-facing acceptance criterion? The record's
   own free-text fields are left as hints below; fill them in with your own
   reasoning, not a restated answer copied from this prompt.
4. Write an `asserted_solution` record `AS-C06-EX`: which mechanism
   `WaterMover` commits to, and which plausible alternative it is selected
   over — mirroring `AS-C06`'s own mechanism-selection reasoning (a
   confirmed model fact, e.g. `BrewStart`/`BrewCancel` resolving via
   `model.find()`, plus a domain premise about how the two mechanisms
   respond to a discrete control signal, not a claim the model already
   connects one and not the other). Only *after* this record is on file,
   build the concrete realization it licenses: `part def Impeller :>
   WaterMover`, mirroring `ResistanceCoil :> HeatGenerator` being built only
   after `AS-C06` argues for it — its own `doc`, if you keep one, should not
   give away this record's conclusion ahead of the record itself. Bind
   `attribute :>> throughput default = ...;` (keep `default` — dropping it
   means `weak`'s own override below cannot bind a second time). Add two
   `Impeller` usages, `rated` and `weak`, mirroring `rated`/`weak :
   ResistanceCoil`: one passing `brewReq` at `Impeller`'s own default, one
   overridden below the threshold with `assert not satisfy brewReq by
   weak;` folded into its own context.
5. Write an `asserted_inference` record `AI-C06-EX`, honestly scoped
   exactly like `AI-C06`: state plainly what this branch's structural
   decomposition establishes and what it does not — **not** a claim that
   the `BrewUnit` decomposition is complete. Cite `perform_relationships`,
   `find_allocations`, and `brewReq(rated)`/`brewReq(weak)` directly in
   `evidence_refs`. Set `premises` to `["AC-C06-EX", "AS-C06-EX",
   "AS-C03-EX", "AI-C04-EX"]`, the coffee-domain analogues of `AI-C06`'s own
   four premises — make sure your own `rationale` actually uses each one,
   not just lists it.

Verify with `model.ok == True` and `validate_record(...) == []` for all
three records.

In [ ]:
import opensysml
from toaster.report import format_diagnostics

conn = opensysml.connect(version="v0.9.0")

# Paste your completed Ch5 model here, then add MoveWater (nested inside
# ApplyWater's own reopened body) and the abstract carrier WaterMover
# (port, unbound throughput slot, perform action).
source = """
# Your solution here
"""

model = conn.load_from_content(source, strict=False)
print(f"Model ok: {model.ok}")
if not model.ok:
    print(format_diagnostics(model.diagnostics))

action = model.find("CoffeeDemo::MoveWater")
if action:
    print(f"MoveWater kind: {action.kind}")


In [ ]:
# Step 2: add BrewAssembly (:> BrewUnit), composing the WaterMover carrier
# with a usage-level allocation, and BrewReq stated on WaterMover itself
# (not BrewUnit).
source_with_decomp = """
# Your solution here (extend source above)
"""

model2 = conn.load_from_content(source_with_decomp, strict=False)
print(f"Model with decomp ok: {model2.ok}")
if not model2.ok:
    print(format_diagnostics(model2.diagnostics))

part_defs = [e.as_dict() for e in model2.query()
             if e.as_dict().get("@type") == "PartDefinition"]
print(f"Part definitions: {[p.get('declaredName') or p.get('name') for p in part_defs]}")


In [ ]:
# Step 3: write the asserted_context framing record.
from toaster.evidence import ReviewRecord, hash_content, validate_record

framing_record = ReviewRecord(
    identifier="AC-C06-EX",
    kind="asserted_context",
    claim="# What brewReq is framed as: a measure of effectiveness (acceptance) or a measure of performance (derived engineering figure), and the model element the framing applies to",
    subject_ref="ToasterDemo::heatGenerationReq",
    model_ref="CoffeeDemo::brewReq",
    content_hash=hash_content(source_with_decomp),
    scope="CoffeeDemo",
    criteria="# What would make the claim MoE versus MoP (see architecture-layers skill), mirroring AC-C06's own criteria for heatGenerationReq",
    premises=[],
    assumption_refs=["# The one modeling judgment this framing rests on"],
    evidence_refs=[
        "CoffeeDemo::BrewReq doc: the rationale states a component "
        "throughput rating threshold, naming no stakeholder acceptance "
        "criterion."
    ],
    rationale="# The argument connecting that evidence to the claim -- why brewReq is (or is not) a MoP, not a MoE",
    counterevidence="# The strongest case for the other framing",
    residual_uncertainties="# State plainly that this split is a contestable modeling judgment, not a settled fact -- what would need to be modeled to derive the threshold instead of asserting it",
    disposition="pending",
    dependency_freshness="current",
    engineering_conclusion="undetermined",
    record_kind="worked_example",
)

framing_errors = validate_record(framing_record)
print(f"AC-C06-EX validation errors: {framing_errors}")


In [ ]:
# Step 4: write the asserted_solution mechanism-selection record.
brew_start = model2.find("CoffeeDemo::BrewStart")
brew_cancel = model2.find("CoffeeDemo::BrewCancel")
print(f"BrewStart: {brew_start.kind}, BrewCancel: {brew_cancel.kind}")

selection_record = ReviewRecord(
    identifier="AS-C06-EX",
    kind="asserted_solution",
    claim="# Which mechanism WaterMover commits to, and which plausible alternative it is selected over",
    subject_ref="ToasterDemo::ResistanceCoil",
    model_ref="CoffeeDemo::WaterMover",
    content_hash=hash_content(source_with_decomp),
    scope="CoffeeDemo::WaterMover and its realizations",
    criteria="# What the chosen mechanism must be able to do: respond to a discrete control signal, expose a rating brewReq can check",
    premises=[
        "CoffeeDemo declares BrewStart and BrewCancel (Chapter 4), "
        "discrete signals marking the start and cancellation of a "
        "brewing cycle, confirmed by model.find() above.",
        "Domain premise, not derived from the model: a motor-driven "
        "impeller responds to being switched on and off directly, "
        "while a gravity-fed drip mechanism has no actuation point of "
        "its own and needs a separate valve or timing mechanism to "
        "start or stop the flow on command. Neither WaterMover nor any "
        "of its realizations is yet connected to a control signal in "
        "this model; this premise is about the physical mechanisms "
        "themselves, not about what the model currently wires "
        "together.",
    ],
    assumption_refs=["# What WaterMover itself does NOT already commit to -- this record is what actually chooses the mechanism"],
    evidence_refs=[
        f"model.find('CoffeeDemo::BrewStart') resolves to a real {brew_start.kind}, confirmed above.",
        f"model.find('CoffeeDemo::BrewCancel') resolves to a real {brew_cancel.kind}, confirmed above.",
    ],
    rationale="# Why this mechanism is preferred, argued from how the two mechanisms work -- not from anything the model currently wires together",
    counterevidence="# What this selection does not rule out, and what stays unmodeled (mirror AS-C06's own challenge about Joule heating)",
    residual_uncertainties="# What would let this selection be revisited against a real trade study",
    disposition="pending",
    dependency_freshness="current",
    engineering_conclusion="undetermined",
    record_kind="worked_example",
)

selection_errors = validate_record(selection_record)
print(f"AS-C06-EX validation errors: {selection_errors}")


In [ ]:
# Step 4 (continued): only now that AS-C06-EX is on record, build the
# concrete realization it licenses. Keep `default` on the :>> throughput
# override below -- without it, `weak`'s own override cannot bind a second
# time and the model will fail to load.
source_with_impeller = """
# Your solution here (extend source_with_decomp above with Impeller :> WaterMover,
# and the rated / weak candidates)
"""

model3 = conn.load_from_content(source_with_impeller, strict=False)
print(f"Model with Impeller ok: {model3.ok}")
if not model3.ok:
    print(format_diagnostics(model3.diagnostics))


In [ ]:
# Step 5: write the stopping judgment, honestly scoped exactly like AI-C06
# -- not a claim that the BrewUnit decomposition is complete.
from toaster.query import find_allocations, perform_relationships

performs = perform_relationships(model3)
allocations = find_allocations(model3)
rated_holds = model3.eval("CoffeeDemo::brewReq(CoffeeDemo::rated)")
weak_holds = model3.eval("CoffeeDemo::brewReq(CoffeeDemo::weak)")
print(f"Perform relationships: {performs}")
print(f"Allocations: {allocations}")
print(f"brewReq(rated) = {rated_holds}, brewReq(weak) = {weak_holds}")

inference_record = ReviewRecord(
    identifier="AI-C06-EX",
    kind="asserted_inference",
    claim="# State plainly what this branch's structural decomposition establishes (MoveWater performed and allocated, brewReq evaluated on two real candidates) and what it does not (BrewUnit's decomposition is NOT complete)",
    subject_ref="ToasterDemo::HeatingAssembly::heatGen",
    model_ref="CoffeeDemo::BrewAssembly::mover",
    content_hash=hash_content(source_with_impeller),
    scope="CoffeeDemo::BrewAssembly::mover and its realizations",
    criteria="# Per the recursion's own stopping rule: performs / connects / verified, condition by condition, for THIS level only -- mirror AI-C06's own criteria exactly",
    premises=["AC-C06-EX", "AS-C06-EX", "AS-C03-EX", "AI-C04-EX"],
    assumption_refs=["# What is printed/loaded above that this record takes as given, without itself re-arguing it"],
    evidence_refs=[
        f"perform_relationships(model): {performs}",
        f"find_allocations(model): {allocations}",
        f"brewReq(rated) = {rated_holds}, brewReq(weak) = {weak_holds}, evaluated directly against the loaded model above.",
    ],
    rationale="# Performs / Connects / Verified, condition by condition -- and make sure each of the four premises actually does work in this paragraph, not just sits in the list",
    counterevidence="# What this branch does NOT show: waterIn unwired, BrewAssembly not composed into any CoffeeMaker candidate, ApplyWater's other flows not decomposed -- mirror AI-C06's own challenge",
    residual_uncertainties="# What is open for whichever exercise takes it up next",
    disposition="pending",
    dependency_freshness="current",
    engineering_conclusion="undetermined",
    record_kind="worked_example",
)

errors = validate_record(inference_record)
print(f"AI-C06-EX validation errors: {errors}")
print(f"Premises: {inference_record.premises}")
conn.close()
